# ELSST Track 1: Test Set Predictions

Generates predictions on the full 1,911 document test set using the best-established
method zero-shot re-ranking, following the shared task's official format for
completeness against the full task specification. Gold labels for this split are not
available, so no quantitative score can be computed or reported here only predictions.

## 1. Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np

project_path = '/content/drive/MyDrive/ELSST_Project'
RESULTS_DIR = os.path.join(project_path, 'results')
CHECKPOINT_DIR = os.path.join(project_path, 'checkpoints')

!pip install google-genai datasets huggingface_hub -q

from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"

from datasets import load_from_disk
from huggingface_hub import hf_hub_download

dataset = load_from_disk(os.path.join(project_path, 'dataset'))

pool_path = hf_hub_download(repo_id="JohnWang10086/elsst-track1", filename="concept_pool.jsonl", repo_type="dataset")
concept_pool_lookup = {}
with open(pool_path, "r") as f:
    for line in f:
        c = json.loads(line)
        concept_pool_lookup[c["concept_id"]] = c

print(f"Loaded {len(concept_pool_lookup)} concepts")

Mounted at /content/drive


concept_pool.jsonl:   0%|          | 0.00/800k [00:00<?, ?B/s]

Loaded 3433 concepts


### 1.1 Inspect the test set

In [2]:
test_rows = dataset['test']
print(f"Test set size: {len(test_rows)}")
print(f"Fields available: {test_rows[0].keys()}")
print(f"\nExample entry:")
print(test_rows[0])

Test set size: 1911
Fields available: dict_keys(['id', 'text', 'document_type', 'generation_labels', 'retrieval_labels'])

Example entry:
{'id': 'test_t00009', 'text': 'Public health standards in the globalized food market refer to the collective legal and technical frameworks designed to ensure that products intended for human consumption are free from substances that could cause illness or injury. As supply chains have expanded to encompass multiple continents, the complexity of maintaining these standards has increased. In the modern era, a single meal may contain ingredients sourced from a dozen different countries, each with varying levels of oversight. This interconnectivity necessitates a rigorous system of checks and balances to prevent the spread of illness and to maintain public trust in the commercial food supply.\n\nThe history of these regulatory systems is rooted in the rapid urbanization of the late 19th and early 20th centuries. Before this period, most consumers purcha

### 1.2 Check for cached concept embeddings
> Checking if the concept embeddings were already cached from an earlier notebook if so, only the 1,911 test documents need encoding, not the full 3,433-concept pool again.

In [3]:
concept_embeddings_path = os.path.join(CHECKPOINT_DIR, 'qwen3_concept_embeddings.npy')
print(f"Cached concept embeddings exist: {os.path.exists(concept_embeddings_path)}")

if os.path.exists(concept_embeddings_path):
    print(f"Size: {os.path.getsize(concept_embeddings_path) / 1e6:.1f} MB")

Cached concept embeddings exist: False


## 2. Qwen3 Top-50 Retrieval for Test Set
> **Approach:** Reproducing the exact Qwen3-Embedding-0.6B method from the baselines notebook same concept_texts construction, same model on the 1,911 test documents, so the shortlist generation step is identical to how validation was retrieved.

### 2.1 Encode test documents
> Encoding the 1,911 test documents with the same Qwen3-Embedding 0.6B model, reusing the cached concept embeddings.

In [4]:
!pip install sentence-transformers faiss-cpu -q

from sentence_transformers import SentenceTransformer
import faiss
import torch, gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

concept_ids = list(concept_pool_lookup.keys())
concept_texts = [f"{c['term']}. {c['definition']}" for c in concept_pool_lookup.values()]

test_ids = [ex['id'] for ex in test_rows]
test_texts = [ex['text'] for ex in test_rows]

qwen_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)

def get_or_encode(name, texts, model, batch_size=32):
    path = os.path.join(CHECKPOINT_DIR, f"{name}.npy")
    if os.path.exists(path):
        print(f"Loaded cached embeddings: {name}")
        return np.load(path)
    print(f"Encoding: {name} ({len(texts)} items)")
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    np.save(path, embeddings)
    return embeddings

gc.collect()
torch.cuda.empty_cache()

qwen_concept_embeddings = get_or_encode("qwen_concept_embeddings", concept_texts, qwen_model)
qwen_test_embeddings = get_or_encode("qwen_test_embeddings", test_texts, qwen_model, batch_size=8)

print("Concept embeddings:", qwen_concept_embeddings.shape)
print("Test embeddings:", qwen_test_embeddings.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.2 MB/s eta 0:00:00
Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Loaded cached embeddings: qwen_concept_embeddings
Encoding: qwen_test_embeddings (1911 items)


Batches:   0%|          | 0/239 [00:00<?, ?it/s]

Concept embeddings: (3433, 1024)
Test embeddings: (1911, 1024)


### 2.2 Build top-50 shortlists
> Same normalized cosine similarity via inner-product FAISS pattern used throughout generating each test document's top-50 candidate concepts for the LLM re-ranking step.

In [5]:
concept_norm = qwen_concept_embeddings / np.linalg.norm(qwen_concept_embeddings, axis=1, keepdims=True)
test_norm = qwen_test_embeddings / np.linalg.norm(qwen_test_embeddings, axis=1, keepdims=True)

index = faiss.IndexFlatIP(concept_norm.shape[1])
index.add(concept_norm.astype('float32'))

top_k = 50
scores, indices = index.search(test_norm.astype('float32'), top_k)

qwen_test_predictions = {
    doc_id: [concept_ids[i] for i in indices[row]]
    for row, doc_id in enumerate(test_ids)
}

with open(os.path.join(RESULTS_DIR, 'qwen_test_predictions.json'), 'w') as f:
    json.dump(qwen_test_predictions, f)

print(f"Generated top-50 shortlists for {len(qwen_test_predictions)} test documents")
print(f"Example - {test_ids[0]}: {[concept_pool_lookup[c]['term'] for c in qwen_test_predictions[test_ids[0]][:5]]}")

Generated top-50 shortlists for 1911 test documents
Example - test_t00009: ['FOOD PREPARATION', 'GLOBAL HEALTH', 'FOOD INSECURITY', 'PRODUCT SAFETY', 'FOOD SECURITY']


## 3. Zero-Shot Re-ranking on Test Set
> **Approach:** Using zero-shot the simplest, most reliable method throughout this project to re-rank each test document's top-50 shortlist, for completeness against the full 1,911-document test set.

### 3.1 Prompt builder
> Reusing the exact zero-shot prompt template from notebook 03, unchanged, so the test-set method matches the validation set method precisely.

In [6]:
def build_zero_shot_prompt(doc_text, candidate_ids):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )
    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named.

Passage:
{doc_text}

Candidate concepts:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""


import re

def parse_ranked_ids(response_text, valid_ids):
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON array found in response: {response_text[:200]}")
    ranked = json.loads(match.group())
    valid_set = set(valid_ids)
    parsed = [cid for cid in ranked if cid in valid_set]
    for cid in valid_ids:
        if cid not in parsed:
            parsed.append(cid)
    return parsed


def call_llm(prompt, max_retries=3, wait_seconds=4):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            time.sleep(wait_seconds)
            return response.text
        except Exception as e:
            if "429" in str(e):
                backoff = 15 * (attempt + 1)
                print(f"Rate limited, waiting {backoff}s (attempt {attempt + 1})...")
                time.sleep(backoff)
            else:
                raise
    raise RuntimeError("Failed after max retries")

### 3.2 Full test set run
> Same checkpointing pattern as every other full run 1,911 documents is over 2.5x the validation set, so this takes longer but needs no active attention.

In [7]:
def compute_test_predictions():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'test_zeroshot_partial.json')
    failed_path = os.path.join(CHECKPOINT_DIR, 'test_zeroshot_failed_ids.json')
    predictions = {}
    failed_ids = []

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(test_ids)} documents already done")

    for i, doc_id in enumerate(test_ids):
        if doc_id in predictions:
            continue

        candidates = qwen_test_predictions[doc_id]
        prompt = build_zero_shot_prompt(test_texts[i], candidates)

        try:
            response_text = call_llm(prompt)
            ranked = parse_ranked_ids(response_text, candidates)
            predictions[doc_id] = ranked
            print(f"[{i+1}/{len(test_ids)}] {doc_id} done")
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
            failed_ids.append(doc_id)

        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            with open(failed_path, 'w') as f:
                json.dump(failed_ids, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(test_ids)} ({len(failed_ids)} failed)")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)
    with open(failed_path, 'w') as f:
        json.dump(failed_ids, f)

    with open(os.path.join(RESULTS_DIR, 'test_zeroshot_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    print(f"\nFinished. {len(failed_ids)}/{len(test_ids)} documents needed fallback.")
    return predictions

test_predictions = compute_test_predictions()

[1/1911] test_t00009 done
Checkpoint saved: 1/1911 (0 failed)
[2/1911] test_t00005 done
[3/1911] test_t00011 done
[4/1911] test_t00001 done
[5/1911] test_t00021 done
[6/1911] test_t00006 done
[7/1911] test_t00020 done
[8/1911] test_t00017 done
[9/1911] test_t00023 done
[10/1911] test_t00003 done
[11/1911] test_t00035 done
[12/1911] test_t00034 done
[13/1911] test_t00002 done
[14/1911] test_t00004 done
[15/1911] test_t00039 done
[16/1911] test_t00024 done
[17/1911] test_t00008 done
[18/1911] test_t00030 done
[19/1911] test_t00037 done
[20/1911] test_t00015 done
[21/1911] test_t00029 done
Checkpoint saved: 21/1911 (0 failed)
[22/1911] test_t00026 done
[23/1911] test_t00038 done
[24/1911] test_t00033 done
[25/1911] test_t00007 done
[26/1911] test_t00019 done
[27/1911] test_t00010 done
[28/1911] test_t00022 done
[29/1911] test_t00012 done
[30/1911] test_t00036 done
[31/1911] test_t00014 done
[32/1911] test_t00018 done
[33/1911] test_t00027 done
[34/1911] test_t00028 done
[35/1911] test_t00

### 3.3 Retry failed documents
> A handful of documents failed with genuine malformed JSON errors during the full run retrying them individually for a clean, complete 1,911/1,911 result.

In [8]:
with open(os.path.join(CHECKPOINT_DIR, 'test_zeroshot_failed_ids.json')) as f:
    test_failed_ids = json.load(f)

print(f"Failed documents: {test_failed_ids}")

for doc_id in test_failed_ids:
    idx = test_ids.index(doc_id)
    candidates = qwen_test_predictions[doc_id]
    prompt = build_zero_shot_prompt(test_texts[idx], candidates)

    try:
        response_text = call_llm(prompt)
        ranked = parse_ranked_ids(response_text, candidates)
        test_predictions[doc_id] = ranked
        print(f"Recovered: {doc_id}")
    except Exception as e:
        print(f"Still failing: {doc_id}: {e}")

with open(os.path.join(RESULTS_DIR, 'test_zeroshot_predictions.json'), 'w') as f:
    json.dump(test_predictions, f)

print(f"\nFinal test set predictions saved: {len(test_predictions)}/{len(test_ids)}")

Failed documents: ['test_t00242', 'test_t00416', 'test_t01495']
Recovered: test_t00242
Recovered: test_t00416
Recovered: test_t01495

Final test set predictions saved: 1911/1911


## 4. Test Set Label Availability
> Checking whether the test split's gold labels are available anywhere in the dataset object, to confirm whether a quantitative score MRR/Recall/NDCG can be computed for these predictions.

In [9]:
labeled_count = sum(1 for row in test_rows if row['retrieval_labels'] is not None)
print(f"Test documents with non-null retrieval_labels: {labeled_count}/{len(test_rows)}")

Test documents with non-null retrieval_labels: 0/1911


## 5. Key Summary

Generated complete predictions for all 1,911 official test-set documents using zero-shot re-ranking 1,911/1,911, zero unresolved fallbacks after retry. The test split's `retrieval_labels` field is `None` for every document 0/1911 labeled gold labels are withheld by the shared task organizers, so no MRR/Recall/NDCG score can be computed or reported for this split. These predictions are included for completeness against the full task specification, following the same method and format as the validation-set results reported elsewhere in this project.